# Lista 1 — MMQ e Integração Numérica

Este notebook foi organizado para rodar diretamente no **Jupyter**.

Estrutura usada em cada questão:
1. **Enunciado reescrito** em linguagem mais direta;
2. **Contexto das funções** utilizadas;
3. **Resolução em Python**, com funções genéricas;
4. **Comentários finais** para interpretação dos resultados.

> Observação importante: em alguns itens o enunciado não fixa todos os detalhes numéricos necessários para uma única resposta. Nesses casos, o notebook deixa a **hipótese adotada explicitamente**.

In [1]:
import numpy as np
import sympy as sp
from math import pi
from numpy.polynomial.legendre import leggauss

np.set_printoptions(precision=8, suppress=True)

## Contexto das funções utilizadas

As funções abaixo foram escritas de forma **genérica**, para que você possa reutilizá-las em outras listas:

- `linear_regression(x, y)`: ajuste linear \(y = a_0 + a_1 x\);
- `log_regression(x, y)`: ajuste logarítmico \(y = a + b\ln(x)\);
- `exp_regression(x, y)`: ajuste exponencial \(y = b e^{mx}\);
- `power_regression(x, y)`: ajuste de potência \(y = b x^m\);
- `poly_regression(x, y, deg)`: regressão polinomial de grau arbitrário;
- `multiple_linear_regression(X, y)`: regressão linear múltipla;
- `composite_trapezoid`, `composite_simpson13`, `composite_simpson38`: integração numérica composta;
- `gauss_quadrature`: quadratura de Gauss-Legendre em intervalo arbitrário.

In [2]:
def _r2(y, yhat):
    y = np.asarray(y, dtype=float)
    yhat = np.asarray(yhat, dtype=float)
    ss_res = np.sum((y - yhat)**2)
    ss_tot = np.sum((y - np.mean(y))**2)
    return 1.0 - ss_res/ss_tot if ss_tot != 0 else 1.0

def linear_regression(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    sx, sy = np.sum(x), np.sum(y)
    sxx, sxy = np.sum(x*x), np.sum(x*y)
    a1 = (n*sxy - sx*sy) / (n*sxx - sx*sx)
    a0 = np.mean(y) - a1*np.mean(x)
    predict = lambda xv: a0 + a1*np.asarray(xv, dtype=float)
    yhat = predict(x)
    return {"modelo": "linear", "a0": a0, "a1": a1, "predict": predict, "r2": _r2(y, yhat), "yhat": yhat}

def log_regression(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if np.any(x <= 0):
        raise ValueError("Modelo logarítmico requer x > 0.")
    base = linear_regression(np.log(x), y)
    a, b = base["a0"], base["a1"]
    predict = lambda xv: a + b*np.log(np.asarray(xv, dtype=float))
    yhat = predict(x)
    return {"modelo": "log", "a": a, "b": b, "predict": predict, "r2": _r2(y, yhat), "yhat": yhat}

def exp_regression(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if np.any(y <= 0):
        raise ValueError("Modelo exponencial requer y > 0.")
    base = linear_regression(x, np.log(y))
    b, m = np.exp(base["a0"]), base["a1"]
    predict = lambda xv: b*np.exp(m*np.asarray(xv, dtype=float))
    yhat = predict(x)
    return {"modelo": "exp", "b": b, "m": m, "predict": predict, "r2": _r2(y, yhat), "yhat": yhat}

def power_regression(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if np.any(x <= 0) or np.any(y <= 0):
        raise ValueError("Modelo de potência requer x > 0 e y > 0.")
    base = linear_regression(np.log(x), np.log(y))
    b, m = np.exp(base["a0"]), base["a1"]
    predict = lambda xv: b*np.asarray(xv, dtype=float)**m
    yhat = predict(x)
    return {"modelo": "potencia", "b": b, "m": m, "predict": predict, "r2": _r2(y, yhat), "yhat": yhat}

def poly_regression(x, y, deg):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    coeff_desc = np.polyfit(x, y, deg)
    poly = np.poly1d(coeff_desc)
    yhat = poly(x)
    return {"modelo": f"polinomio_grau_{deg}", "coeff_desc": coeff_desc, "predict": poly, "r2": _r2(y, yhat), "yhat": yhat}

def multiple_linear_regression(X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    Xaug = np.column_stack([np.ones(len(X)), X])
    beta = np.linalg.lstsq(Xaug, y, rcond=None)[0]
    predict = lambda Xnew: np.column_stack([np.ones(len(np.asarray(Xnew))), np.asarray(Xnew, dtype=float)]) @ beta
    yhat = Xaug @ beta
    return {"beta": beta, "predict": predict, "r2": _r2(y, yhat), "yhat": yhat}

def composite_trapezoid(f, a, b, n):
    x = np.linspace(a, b, n + 1)
    y = f(x)
    h = (b - a) / n
    return h * (0.5*y[0] + np.sum(y[1:-1]) + 0.5*y[-1])

def composite_simpson13(f, a, b, n):
    if n % 2 != 0:
        raise ValueError("Simpson 1/3 composto requer n par.")
    x = np.linspace(a, b, n + 1)
    y = f(x)
    h = (b - a) / n
    return h/3 * (y[0] + y[-1] + 4*np.sum(y[1:-1:2]) + 2*np.sum(y[2:-1:2]))

def composite_simpson38(f, a, b, n):
    if n % 3 != 0:
        raise ValueError("Simpson 3/8 composto requer n múltiplo de 3.")
    x = np.linspace(a, b, n + 1)
    y = f(x)
    h = (b - a) / n
    total = y[0] + y[-1]
    for k in range(1, n):
        total += (2 if k % 3 == 0 else 3) * y[k]
    return 3*h*total/8

def gauss_quadrature(f, a, b, n):
    xi, wi = leggauss(n)
    xm = 0.5*(b - a)*xi + 0.5*(a + b)
    return 0.5*(b - a) * np.sum(wi * f(xm))

def print_model_result(nome, fit, x_pred=None):
    print(f"\n{nome}")
    print("-"*len(nome))
    for k, v in fit.items():
        if k not in {"predict", "yhat"}:
            print(f"{k}: {v}")
    if x_pred is not None:
        print(f"predição em x = {x_pred}: {fit['predict'](x_pred)}")

## Questão 1

**Enunciado reescrito.** Ajustar os dados de corrente \(i\) e queda de voltagem \(V\) usando os modelos de MMQ que façam sentido e estimar \(V(1{,}15)\).

**Leitura matemática do problema.**
Como existem valores de tensão negativos, os modelos que exigem \(y>0\) em toda a amostra podem ficar inviáveis.

In [3]:
i = np.array([0.25, 0.75, 1.25, 1.50, 2.00], dtype=float)
V = np.array([-0.45, -0.60, 0.70, 1.88, 6.00], dtype=float)

ajustes = {}

ajustes["linear"] = linear_regression(i, V)
ajustes["logaritmico"] = log_regression(i, V)
ajustes["quadratico"] = poly_regression(i, V, 2)

for modelo in ["exponencial", "potencia"]:
    try:
        if modelo == "exponencial":
            ajustes[modelo] = exp_regression(i, V)
        else:
            ajustes[modelo] = power_regression(i, V)
    except ValueError as e:
        ajustes[modelo] = str(e)

for nome, fit in ajustes.items():
    if isinstance(fit, dict):
        print_model_result(nome, fit, x_pred=1.15)
    else:
        print(f"\n{nome}\n{'-'*len(nome)}\n{fit}")

print("\nResumo interpretativo:")
print("- Modelos exponencial e de potência não podem ser aplicados diretamente porque a amostra tem y <= 0.")
print("- Entre os modelos viáveis, o quadrático apresenta o maior R² e a melhor aderência aos dados.")


linear
------
modelo: linear
a0: -2.572876712328766
a1: 3.5468493150684925
r2: 0.7850027495882735
predição em x = 1.15: 1.5059999999999998

logaritmico
-----------
modelo: log
a: 1.6746882177684252
b: 2.3946387695475875
r2: 0.5243613887124325
predição em x = 1.15: 2.009367583487256

quadratico
----------
modelo: polinomio_grau_2
coeff_desc: [ 3.39445474 -4.00929994  0.38855255]
r2: 0.9988106057535018
predição em x = 1.15: 0.2670240189311784

exponencial
-----------
Modelo exponencial requer y > 0.

potencia
--------
Modelo de potência requer x > 0 e y > 0.

Resumo interpretativo:
- Modelos exponencial e de potência não podem ser aplicados diretamente porque a amostra tem y <= 0.
- Entre os modelos viáveis, o quadrático apresenta o maior R² e a melhor aderência aos dados.


**Comentário.**  
Para este conjunto, o melhor ajuste entre os modelos viáveis ficou com o **polinômio de 2º grau**.  
A estimativa mais consistente para \(V(1{,}15)\) é a do ajuste quadrático.

## Questão 2

**Enunciado reescrito.** Usar **duas funções** de regressão por MMQ no conjunto dado e interpretar.

**Escolha adotada aqui.** Vou comparar:
- modelo **linear**;
- modelo **quadrático**.

Essa comparação é útil porque o conjunto parece ter uma tendência global decrescente, mas com curvatura.

In [4]:
x = np.array([6, 7, 11, 15, 17, 21, 23, 29, 33, 37, 39], dtype=float)
y = np.array([29, 21, 29, 14, 21, 15, 7, 7, 13, 0, 3], dtype=float)

fit_lin = linear_regression(x, y)
fit_quad = poly_regression(x, y, 2)

print_model_result("Ajuste linear", fit_lin)
print_model_result("Ajuste quadrático", fit_quad)

print("\nInterpretação:")
print(f"- R² linear     = {fit_lin['r2']:.6f}")
print(f"- R² quadrático = {fit_quad['r2']:.6f}")
print("- O ajuste quadrático é ligeiramente melhor, mas a melhora é pequena.")


Ajuste linear
-------------
modelo: linear
a0: 30.48737137511693
a1: -0.7410129627154884
r2: 0.7759994104357664

Ajuste quadrático
-----------------
modelo: polinomio_grau_2
coeff_desc: [ 0.00751415 -1.07819264 33.33570426]
r2: 0.7827640088707885

Interpretação:
- R² linear     = 0.775999
- R² quadrático = 0.782764
- O ajuste quadrático é ligeiramente melhor, mas a melhora é pequena.


## Questão 3

**Enunciado reescrito.** Ajustar os dados pelos modelos:
1. linear;
2. potência;
3. polinomial de 2º grau.

Depois, comparar os resultados.

In [5]:
x = np.array([5, 10, 15, 20, 25, 30, 35, 40, 45, 50], dtype=float)
y = np.array([17, 24, 31, 33, 37, 37, 40, 40, 42, 41], dtype=float)

fit_linear = linear_regression(x, y)
fit_power = power_regression(x, y)
fit_quad = poly_regression(x, y, 2)

print_model_result("Modelo linear", fit_linear)
print_model_result("Modelo de potência", fit_power)
print_model_result("Modelo quadrático", fit_quad)

print("\nEquações ajustadas:")
print(f"Linear:      y = {fit_linear['a0']:.6f} + {fit_linear['a1']:.6f} x")
print(f"Potência:    y = {fit_power['b']:.6f} x^({fit_power['m']:.6f})")
c2, c1, c0 = fit_quad['coeff_desc']
print(f"Quadrático:  y = {c2:.6f} x² + {c1:.6f} x + {c0:.6f}")

print("\nConclusão: o melhor R² entre os três ajustes indica o melhor modelo para esses dados.")


Modelo linear
-------------
modelo: linear
a0: 20.6
a1: 0.49454545454545457
r2: 0.8384912959381045

Modelo de potência
------------------
modelo: potencia
b: 9.952915111289023
m: 0.38507720531853296
r2: 0.9377000865888203

Modelo quadrático
-----------------
modelo: polinomio_grau_2
coeff_desc: [-0.01606061  1.37787879 11.76666667]
r2: 0.9799826724693745

Equações ajustadas:
Linear:      y = 20.600000 + 0.494545 x
Potência:    y = 9.952915 x^(0.385077)
Quadrático:  y = -0.016061 x² + 1.377879 x + 11.766667

Conclusão: o melhor R² entre os três ajustes indica o melhor modelo para esses dados.


## Questão 4

**Enunciado reescrito.** Ajustar os dados do túnel de vento pelos modelos:
- logarítmico;
- potência;
- exponencial;

e analisar qual descreve melhor a relação entre velocidade do vento e força.

In [6]:
v = np.array([10, 20, 30, 40, 50, 60, 70, 80], dtype=float)
F = np.array([25, 70, 380, 550, 610, 1220, 830, 1450], dtype=float)

fit_log = log_regression(v, F)
fit_pow = power_regression(v, F)
fit_exp = exp_regression(v, F)

print_model_result("Modelo logarítmico", fit_log)
print_model_result("Modelo de potência", fit_pow)
print_model_result("Modelo exponencial", fit_exp)

print("\nEquações ajustadas:")
print(f"Logarítmico: F = {fit_log['a']:.6f} + {fit_log['b']:.6f} ln(v)")
print(f"Potência:    F = {fit_pow['b']:.6f} v^({fit_pow['m']:.6f})")
print(f"Exponencial: F = {fit_exp['b']:.6f} e^({fit_exp['m']:.6f} v)")

print("\nInterpretação:")
melhor = max([fit_log, fit_pow, fit_exp], key=lambda d: d['r2'])
print(f"- O maior R² ficou com o modelo: {melhor['modelo']}")


Modelo logarítmico
------------------
modelo: log
a: -1683.3753420658668
b: 640.8896106909673
r2: 0.7866708409181533

Modelo de potência
------------------
modelo: potencia
b: 0.2741373420132226
m: 1.9841762557640112
r2: 0.8088181209722812

Modelo exponencial
------------------
modelo: exp
b: 34.01208800674337
m: 0.052845950835110435
r2: 0.23732796628920416

Equações ajustadas:
Logarítmico: F = -1683.375342 + 640.889611 ln(v)
Potência:    F = 0.274137 v^(1.984176)
Exponencial: F = 34.012088 e^(0.052846 v)

Interpretação:
- O maior R² ficou com o modelo: potencia


## Questão 5

**Enunciado reescrito.** Ajustar um modelo de **regressão linear múltipla**
\[
y = a_0 + a_1 x_1 + a_2 x_2
\]
para os dados fornecidos.

In [7]:
x1 = np.array([0, 1, 1, 2, 2, 3, 3, 4, 4], dtype=float)
x2 = np.array([0, 1, 2, 1, 2, 1, 2, 1, 2], dtype=float)
y  = np.array([15.1, 17.9, 12.7, 25.6, 20.5, 35.1, 29.7, 45.4, 40.2], dtype=float)

fit = multiple_linear_regression(np.column_stack([x1, x2]), y)
a0, a1, a2 = fit["beta"]

print(f"Coeficientes ajustados:")
print(f"a0 = {a0:.6f}")
print(f"a1 = {a1:.6f}")
print(f"a2 = {a2:.6f}")
print(f"R² = {fit['r2']:.6f}")
print(f"Modelo: y = {a0:.6f} + {a1:.6f} x1 + {a2:.6f} x2")

Coeficientes ajustados:
a0 = 14.460870
a1 = 9.025217
a2 = -5.704348
R² = 0.995523
Modelo: y = 14.460870 + 9.025217 x1 + -5.704348 x2


## Questão 6

**Enunciado reescrito.** Em um circuito RL série, calcular a energia dissipada no resistor
\[
W_R = \int_0^2 R[i(t)]^2 \, dt
\]
usando trapézio, Simpson 1/3 e Simpson 3/8, em três malhas numéricas diferentes.

**Hipótese adotada.** Como o enunciado não fornece condição inicial, vou usar a hipótese física mais comum em problemas de chaveamento:
\[
i(0) = 0.
\]

Também vou comparar os métodos para:
\[
n = 6,\ 12,\ 24
\]
pois esses valores funcionam simultaneamente para os três métodos.

In [8]:
t = sp.symbols('t', real=True)
i = sp.Function('i')

ode = sp.Eq(sp.diff(i(t), t) + 1000*i(t), 100*sp.exp(-2*t)*sp.sin(5*t))
sol = sp.dsolve(ode, ics={i(0): 0})
i_expr = sp.simplify(sol.rhs)

print("Corrente i(t):")
display(sol)

i_num = sp.lambdify(
    t,
    sp.exp(-2*t)*(sp.Rational(99800, 996029)*sp.sin(5*t)
                  - sp.Rational(500, 996029)*sp.cos(5*t)
                  + sp.Rational(500, 996029)*sp.exp(-998*t)),
    'numpy'
)

R = 100.0
g = lambda tt: R*(i_num(tt)**2)

W_exact = sp.N(sp.integrate(R*i_expr**2, (t, 0, 2)), 20)
print(f"Energia de referência (integração simbólica): {float(W_exact):.12f} J")

for n in [6, 12, 24]:
    trap = composite_trapezoid(g, 0, 2, n)
    s13 = composite_simpson13(g, 0, 2, n)
    s38 = composite_simpson38(g, 0, 2, n)
    print(f"\nSubintervalos n = {n}")
    print(f"Trapézio       = {trap:.12f} | erro = {abs(trap - float(W_exact)):.12e}")
    print(f"Simpson 1/3    = {s13:.12f} | erro = {abs(s13 - float(W_exact)):.12e}")
    print(f"Simpson 3/8    = {s38:.12f} | erro = {abs(s38 - float(W_exact)):.12e}")
    print(f"Avaliações de f = {n+1}")

Corrente i(t):


Eq(i(t), (99800*sin(5*t)/996029 - 500*cos(5*t)/996029 + 500*exp(-998*t)/996029)*exp(-2*t))

Energia de referência (integração simbólica): 0.107702637488 J

Subintervalos n = 6
Trapézio       = 0.094518098392 | erro = 1.318453909610e-02
Simpson 1/3    = 0.125331874714 | erro = 1.762923722546e-02
Simpson 3/8    = 0.104206881619 | erro = 3.495755869248e-03
Avaliações de f = 7

Subintervalos n = 12
Trapézio       = 0.107126883849 | erro = 5.757536394093e-04
Simpson 1/3    = 0.111329812335 | erro = 3.627174846155e-03
Simpson 3/8    = 0.116235922062 | erro = 8.533284573455e-03
Avaliações de f = 13

Subintervalos n = 24
Trapézio       = 0.107690474299 | erro = 1.216318992445e-05
Simpson 1/3    = 0.107878337782 | erro = 1.757002932371e-04
Simpson 3/8    = 0.108136670627 | erro = 4.340331383634e-04
Avaliações de f = 25


## Questão 7

**Enunciado reescrito.** Comparar a quadratura de Gauss usando 2, 4 e 6 pontos para
\[
\int_0^3 x e^x \, dx.
\]

Vou usar como referência o valor exato:
\[
\int x e^x dx = (x-1)e^x.
\]

In [9]:
f = lambda x: x*np.exp(x)
I_exact = 2*np.e**3 + 1
print(f"Valor exato = {I_exact:.12f}")

for n in [2, 4, 6]:
    val = gauss_quadrature(f, 0, 3, n)
    print(f"n = {n}: {val:.12f} | erro = {abs(val - I_exact):.12e}")

Valor exato = 41.171073846375
n = 2: 39.607502004045 | erro = 1.563571842331e+00
n = 4: 41.170569916547 | erro = 5.039298286604e-04
n = 6: 41.171073827382 | erro = 1.899341128819e-08


## Questão 8

**Enunciado reescrito.** Determinar a distância percorrida pelo avião até reduzir a velocidade de 93 m/s para 40 m/s, usando o **método trapezoidal composto**.

Da relação
\[
m v \frac{dv}{dx} = -5v^2 - 570000
\]
fazemos separação de variáveis:
\[
dx = -\frac{97000\,v}{5v^2 + 570000} dv.
\]

Como a velocidade cai de 93 para 40 m/s, a distância positiva fica:
\[
x = \int_{40}^{93} \frac{97000\,v}{5v^2 + 570000} dv.
\]

Como o enunciado não fixa o número de subintervalos, vou mostrar uma **tabela de convergência**.

In [10]:
integrand = lambda v: 97000*v/(5*v*v + 570000)

# referência numérica alta precisão
x_ref = gauss_quadrature(integrand, 40, 93, 20)
print(f"Referência numérica (Gauss 20 pontos): {x_ref:.12f} m")

for n in [10, 20, 50, 100, 200, 500, 1000]:
    val = composite_trapezoid(integrand, 40, 93, n)
    print(f"n = {n:4d}: distância = {val:.12f} m | erro ≈ {abs(val - x_ref):.12e}")

Referência numérica (Gauss 20 pontos): 574.149413167485 m
n =   10: distância = 574.085485133712 m | erro ≈ 6.392803377298e-02
n =   20: distância = 574.133431997369 m | erro ≈ 1.598117011611e-02
n =   50: distância = 574.146856217817 m | erro ≈ 2.556949668474e-03
n =  100: distância = 574.148773931409 m | erro ≈ 6.392360762675e-04
n =  200: distância = 574.149253358550 m | erro ≈ 1.598089351091e-04
n =  500: distância = 574.149387598060 m | erro ≈ 2.556942581577e-05
n = 1000: distância = 574.149406775129 m | erro ≈ 6.392356567630e-06


## Questão 9

**Enunciado reescrito.** Usar Simpson 1/3 composto para estimar a **área superficial** e o **volume** de uma bola de futebol americano a partir dos dados tabelados.

As fórmulas fornecidas são:
\[
S = 2\pi \int_0^L r\,dz,
\qquad
V = \pi \int_0^L r^2\,dz.
\]

Como a tabela apresenta valores de \(z\) arredondados, adoto o passo uniforme reconstruído a partir dos extremos:
\[
h = \frac{30{,}5 - 0}{12}.
\]

In [11]:
z = np.array([0, 2.5, 5.1, 7.6, 10.2, 12.7, 15.2, 17.8, 20.3, 22.9, 25.4, 27.9, 30.5], dtype=float)
d = np.array([0, 6.6, 8.1, 12.2, 14.2, 15.2, 15.7, 15.2, 14.2, 12.2, 8.1, 6.6, 0], dtype=float)

r = d / 2.0
h = (z[-1] - z[0]) / (len(z) - 1)

def simpson13_tabulated(y, h):
    n = len(y) - 1
    if n % 2 != 0:
        raise ValueError("A tabela precisa ter número par de subintervalos.")
    return h/3 * (y[0] + y[-1] + 4*np.sum(y[1:-1:2]) + 2*np.sum(y[2:-1:2]))

S = 2*np.pi*simpson13_tabulated(r, h)
V = np.pi*simpson13_tabulated(r**2, h)

print(f"h reconstruído = {h:.6f} cm")
print(f"Área superficial estimada S = {S:.6f} cm²")
print(f"Volume estimado V           = {V:.6f} cm³")

h reconstruído = 2.541667 cm
Área superficial estimada S = 1044.954803 cm²
Volume estimado V           = 3293.430844 cm³


## Questão 10

**Enunciado reescrito.** Determinar o comprimento do arco do Gateway Arch, modelado por
\[
f(x) = 211{,}5 - 20{,}97\cosh\left(\frac{x}{30{,}38}\right),
\qquad -91{,}21 \le x \le 91{,}21
\]
usando:
- Simpson 1/3 com 8 subintervalos;
- Simpson 3/8 com 9 subintervalos;
- quadratura de Gauss de segunda ordem.

A integral do comprimento de arco é:
\[
L = \int_a^b \sqrt{1 + [f'(x)]^2}\,dx.
\]

In [12]:
a, b = -91.21, 91.21
fprime = lambda x: -(20.97/30.38) * np.sinh(np.asarray(x, dtype=float)/30.38)
integrand = lambda x: np.sqrt(1 + fprime(x)**2)

# referência numérica precisa
L_ref = gauss_quadrature(integrand, a, b, 40)

L_s13 = composite_simpson13(integrand, a, b, 8)
L_s38 = composite_simpson38(integrand, a, b, 9)
L_g2  = gauss_quadrature(integrand, a, b, 2)

print(f"Referência numérica (Gauss 40 pontos): {L_ref:.12f}")
print(f"Simpson 1/3 (n=8):  {L_s13:.12f} | erro ≈ {abs(L_s13 - L_ref):.12e}")
print(f"Simpson 3/8 (n=9):  {L_s38:.12f} | erro ≈ {abs(L_s38 - L_ref):.12e}")
print(f"Gauss 2 pontos:     {L_g2:.12f} | erro ≈ {abs(L_g2 - L_ref):.12e}")

print("\nComentário:")
print("- As fórmulas de Simpson, com vários subintervalos, capturam melhor a curvatura do arco.")
print("- A quadratura de Gauss de ordem 2 usa poucos pontos; por isso aqui ela perde precisão.")

Referência numérica (Gauss 40 pontos): 451.397358494416
Simpson 1/3 (n=8):  452.090612135377 | erro ≈ 6.932536409611e-01
Simpson 3/8 (n=9):  452.364763356308 | erro ≈ 9.674048618916e-01
Gauss 2 pontos:     390.440180939985 | erro ≈ 6.095717755443e+01

Comentário:
- As fórmulas de Simpson, com vários subintervalos, capturam melhor a curvatura do arco.
- A quadratura de Gauss de ordem 2 usa poucos pontos; por isso aqui ela perde precisão.
